In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
%matplotlib inline 


In [3]:
ratings_data = pd.read_csv("dataset/ratings.csv")
ratings_data = ratings_data.drop('timestamp', axis = 1)
ratings_data.head()

,userId,movieId,rating
0,1,110,1.0
1,1,147,4.5
2,1,858,5.0
3,1,1221,5.0
4,1,1246,5.0


In [4]:
ratings_data.shape

(26024289, 3)

In [5]:
movie_names = pd.read_csv("dataset/movies_metadata.csv")
movie_names = movie_names[['title', 'id']]
movie_names.head()



/var/folders/5n/10j2bk_d6_5d_78gychkbs6m0000gn/T/ipykernel_474/1099518790.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_names = pd.read_csv("dataset/movies_metadata.csv")


,title,id
0,Toy Story,862
1,Jumanji,8844
2,Grumpier Old Men,15602
3,Waiting to Exhale,31357
4,Father of the Bride Part II,11862


In [6]:
movie_names.shape

(45466, 2)

In [7]:
links = pd.read_csv("dataset/links.csv")
#movie_names = movie_names[['title', 'genres']]
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [8]:
movie_names['id'] = pd.to_numeric(movie_names['id'], errors='coerce')
movie_names = movie_names.drop_duplicates(subset='id')

ratings_with_tmdb = ratings_data.merge(links[['movieId', 'tmdbId']], on='movieId', how='left')
merged = ratings_with_tmdb.merge(movie_names[['id', 'title']], left_on='tmdbId', right_on='id', how='left')

lookup_title = merged[['movieId', 'title']].dropna().drop_duplicates()

lookup_title.head()

,movieId,title
0,110,Braveheart
1,147,The Basketball Diaries
2,858,The Godfather
3,1221,The Godfather: Part II
4,1246,Dead Poets Society


In [9]:
lookup_title.shape

(44738, 2)

In [10]:
title = lookup_title.loc[lookup_title['movieId'] == 31, 'title'].values[0]
print(title)

Dangerous Minds


### Create the matrix

In [13]:
user_ids = ratings_data['userId'].astype('category')
movie_ids = ratings_data['movieId'].astype('category')

# Build the sparse matrix
movies_users = csr_matrix(
    (ratings_data['rating'].values,
     (user_ids.cat.codes, movie_ids.cat.codes))
)

user_index_to_id = dict(enumerate(user_ids.cat.categories))
movie_index_to_id = dict(enumerate(movie_ids.cat.categories))

In [14]:
svd = TruncatedSVD(n_components=20, random_state=42) 
movie_factors = svd.fit_transform(movies_users.T) 

movie_ids_ordered = pd.Index(movie_index_to_id.values())  # These match the column indices of the sparse matrix
movie_factors_df = pd.DataFrame(movie_factors, index=movie_ids_ordered)

In [15]:
movie_factors_df.to_csv('collaborative_matrix.csv', index=True)

In [16]:
lookup_title.to_csv('lookup_title.csv', index=False)